In [1]:
import numpy as np
from jaxtyping import Float
import pandas as pd
import matplotlib.pyplot as plt

# muutils
from muutils.jsonlines import jsonl_write, jsonl_load

# attention-motifs
from attention_motifs.bins import Bins
from attention_motifs.features.features import scalar_feature_table
from attention_motifs.features.hist_beta_fit import hist_beta_fit
from attention_motifs.util import prefix_dict
from attention_motifs.features.transition_tensor import tt_features
from attention_motifs.features.vec_features import vec_features
from attention_motifs.math.cos_sim import cosine_similarity_matrix
from attention_motifs.math.math import skew_lt

In [2]:
def gram_features(A: Float[np.ndarray, "n_ctx n_ctx"]) -> dict[str, float]:
	# dbg_tensor(A)
	return prefix_dict(
		hist_beta_fit(
			A.flatten(),
			bins=Bins(n_bins=32, start=0.0, stop=1.0),
		),
		prefix="beta_hist",
	)
	# TODO: mass as a function of distance from diagonal

In [3]:
def compute_scalar_features(
	A: Float[np.ndarray, "n_ctx n_ctx"],
) -> dict[str, float]:
	# dbg_tensor(A)
	A_log: Float[np.ndarray, "n_ctx n_ctx"] = np.nan_to_num(np.log(A + 1e-9), nan=-10)
	# dbg_tensor(A_log)

	A_skew: Float[np.ndarray, "n_ctx n_ctx"] = skew_lt(A)
	# dbg_tensor(A_skew)
	A_log_skew: Float[np.ndarray, "n_ctx n_ctx"] = skew_lt(A_log)

	return dict(
		# diagonal: standard features, fit diff to beta dist
		**prefix_dict(vec_features(A.diagonal()), prefix="diag"),
		# off-diagonal: standard features, fit diff to beta dist
		**prefix_dict(vec_features(A[:, 0]), prefix="first_tok"),
		# transition tensor: standard features, standard features on diff, linear envelope on transition time
		# 	TODO: standard features on decay rate
		**prefix_dict(
			tt_features(A),
			prefix="markov_transition",
		),
		# # {log, raw} gram matrix of {rows, cols, rows of skewed}: beta fit hist
		# # 	TODO: fit fft in `gram_features`, but this is expensive
		**prefix_dict(
			gram_features(A @ A.T),
			prefix=["gram", "row"],
		),
		**prefix_dict(
			gram_features(A.T @ A),
			prefix=["gram", "col"],
		),
		**prefix_dict(
			gram_features(A_skew.T @ A_skew),
			prefix=["gram", "skew"],
		),
		**prefix_dict(
			gram_features(cosine_similarity_matrix(A_log)),
			prefix=["log", "gram", "row"],
		),
		**prefix_dict(
			gram_features(cosine_similarity_matrix(A_log, col=True)),
			prefix=["log", "gram", "col"],
		),
		**prefix_dict(
			gram_features(cosine_similarity_matrix(A_log_skew)),
			prefix=["log", "gram", "skew"],
		),
	)

In [4]:
df: pd.DataFrame = scalar_feature_table(
	features_func=compute_scalar_features,
	# models="pythia-14m".split(","),
)

models: ['pythia-14m', 'gemma-2b', 'gpt2-small', 'pythia-1b', 'gpt2-medium', 'tiny-stories-1M']
model: 'pythia-14m'
✔️  (0.00s) setting up paths                                                   
✔️  (0.00s) loading prompts                                                    
128 prompts loaded


100%|██████████| 128/128 [01:03<00:00,  2.02it/s]


model: 'gemma-2b'
✔️  (0.00s) setting up paths                                                   
✔️  (0.01s) loading prompts                                                    
128 prompts loaded


100%|██████████| 128/128 [05:34<00:00,  2.61s/it]


model: 'gpt2-small'
✔️  (0.00s) setting up paths                                                   
✔️  (0.00s) loading prompts                                                    
142 prompts loaded


...

100%|██████████| 128/128 [05:21<00:00,  2.51s/it]


model: 'gpt2-medium'
✔️  (0.00s) setting up paths                                                   
✔️  (0.01s) loading prompts                                                    
128 prompts loaded


100%|██████████| 128/128 [20:27<00:00,  9.59s/it] 


model: 'tiny-stories-1M'
✔️  (0.00s) setting up paths                                                   
✔️  (0.01s) loading prompts                                                    
128 prompts loaded


100%|██████████| 128/128 [07:00<00:00,  3.29s/it] 


In [5]:
path: str = "../data/scalar_features.jsonl.gz"


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123872 entries, 0 to 123871
Columns: 182 entries, activation.model to feat.log.gram.skew.beta_hist.hist.raw.linreg.r2
dtypes: float32(14), float64(162), int64(2), object(4)
memory usage: 165.4+ MB


In [7]:
df.dtypes

activation.model                                           object
activation.layer                                            int64
activation.cache_key                                       object
activation.head                                             int64
activation.cls                                             object
                                                           ...   
feat.log.gram.skew.beta_hist.hist.raw.autocorr_lag1       float64
feat.log.gram.skew.beta_hist.hist.raw.psd_total_power     float64
feat.log.gram.skew.beta_hist.hist.raw.linreg.slope        float64
feat.log.gram.skew.beta_hist.hist.raw.linreg.intercept    float64
feat.log.gram.skew.beta_hist.hist.raw.linreg.r2           float64
Length: 182, dtype: object

In [8]:

jsonl_write(path, df.to_dict(orient="records"), use_gzip=True)

In [9]:
_temp_loaded = pd.DataFrame(jsonl_load(path))
df = _temp_loaded

In [10]:
df.head()

,activation.model,activation.layer,activation.cache_key,activation.head,activation.cls,activation.prompt,feat.diag.mean,feat.diag.median,feat.diag.variance,feat.diag.std,...,feat.log.gram.skew.beta_hist.hist.raw.L1_norm,feat.log.gram.skew.beta_hist.hist.raw.L2_norm,feat.log.gram.skew.beta_hist.hist.raw.rms,feat.log.gram.skew.beta_hist.hist.raw.energy,feat.log.gram.skew.beta_hist.hist.raw.zero_crossing_rate,feat.log.gram.skew.beta_hist.hist.raw.autocorr_lag1,feat.log.gram.skew.beta_hist.hist.raw.psd_total_power,feat.log.gram.skew.beta_hist.hist.raw.linreg.slope,feat.log.gram.skew.beta_hist.hist.raw.linreg.intercept,feat.log.gram.skew.beta_hist.hist.raw.linreg.r2
0,pythia-14m,0,blocks.0.attn.hook_pattern,0,pythia-14m:L0:H0,MjWkadUxH4Z5nfLEFtUc1w,0.169934,0.127726,0.027237,0.165037,...,0.931865,0.557072,3.151275,317.777149,0.0,-0.025481,16.549212,-0.055552,1.792928,0.029031
1,pythia-14m,0,blocks.0.attn.hook_pattern,1,pythia-14m:L0:H1,MjWkadUxH4Z5nfLEFtUc1w,0.251202,0.123246,0.080360,0.283479,...,1.087790,0.560043,3.168083,321.176074,0.0,-0.198448,14.921705,-0.067310,2.131102,0.043626
2,pythia-14m,0,blocks.0.attn.hook_pattern,2,pythia-14m:L0:H2,MjWkadUxH4Z5nfLEFtUc1w,0.042076,0.025520,0.009217,0.096006,...,0.861793,0.555911,3.144708,316.453958,0.0,0.003151,18.221980,-0.058307,1.765544,0.031686
3,pythia-14m,0,blocks.0.attn.hook_pattern,3,pythia-14m:L0:H3,MjWkadUxH4Z5nfLEFtUc1w,0.151129,0.119528,0.019528,0.139741,...,0.949819,0.557630,3.154430,318.413779,0.0,-0.041139,15.016021,-0.056190,1.820761,0.029747
4,pythia-14m,1,blocks.1.attn.hook_pattern,0,pythia-14m:L1:H0,MjWkadUxH4Z5nfLEFtUc1w,0.644803,0.731561,0.072123,0.268558,...,1.246213,0.565048,3.196392,326.941453,0.0,-0.284769,8.932156,-0.078382,2.461131,0.060452


In [11]:
df.describe()

,activation.layer,activation.head,feat.diag.mean,feat.diag.median,feat.diag.variance,feat.diag.std,feat.diag.skewness,feat.diag.kurtosis,feat.diag.entropy,feat.diag.L1_norm,...,feat.log.gram.skew.beta_hist.hist.raw.L1_norm,feat.log.gram.skew.beta_hist.hist.raw.L2_norm,feat.log.gram.skew.beta_hist.hist.raw.rms,feat.log.gram.skew.beta_hist.hist.raw.energy,feat.log.gram.skew.beta_hist.hist.raw.zero_crossing_rate,feat.log.gram.skew.beta_hist.hist.raw.autocorr_lag1,feat.log.gram.skew.beta_hist.hist.raw.psd_total_power,feat.log.gram.skew.beta_hist.hist.raw.linreg.slope,feat.log.gram.skew.beta_hist.hist.raw.linreg.intercept,feat.log.gram.skew.beta_hist.hist.raw.linreg.r2
count,123872.000000,123872.000000,123872.000000,1.238720e+05,1.238720e+05,123872.000000,123872.000000,123872.000000,123872.000000,123872.000000,...,123872.000000,123872.000000,123872.000000,123872.000000,123872.0,123872.000000,123872.000000,123872.000000,123872.000000,1.238720e+05
mean,8.252777,5.896797,0.086008,6.005637e-02,1.604102e-02,0.120131,6.691383,58.875411,0.767775,0.086008,...,1.013209,0.663897,3.755569,551.920718,0.0,-0.017712,25.305828,-0.078086,2.223538,3.546360e-02
std,6.155531,4.265711,0.121285,1.283139e-01,1.434515e-02,0.040119,3.048786,41.070518,0.709781,0.121285,...,0.321259,0.313411,1.772921,676.670806,0.0,0.063111,25.183312,0.058087,1.204458,1.505247e-02
min,0.000000,0.000000,0.004193,3.548088e-42,5.844798e-07,0.000765,-10.859128,-1.971993,0.033173,0.004193,...,0.460923,0.234712,1.327731,56.411789,0.0,-0.611463,0.582625,-0.454373,0.483393,1.297980e-09
25%,3.000000,2.000000,0.030995,1.092215e-02,9.259977e-03,0.096229,4.680123,26.542540,0.227392,0.030995,...,0.832584,0.520279,2.943142,277.186649,0.0,-0.052184,14.531532,-0.080427,1.655927,2.818638e-02
50%,7.000000,5.000000,0.051313,2.442243e-02,1.146284e-02,0.107065,6.972927,55.875981,0.519150,0.051313,...,0.922654,0.575769,3.257041,339.466085,0.0,-0.015910,18.080705,-0.062300,1.886758,3.381340e-02
75%,12.000000,9.000000,0.089429,5.277387e-02,1.803780e-02,0.134305,9.031805,86.046488,1.084543,0.089429,...,1.065566,0.677043,3.829935,469.388759,0.0,0.008436,23.517405,-0.051215,2.278211,4.116722e-02
max,23.000000,15.000000,0.999533,9.998407e-01,2.475118e-01,0.497506,16.795980,281.346027,3.293428,0.999533,...,3.160517,2.600747,14.712046,6926.217306,0.0,0.922468,305.158755,0.010628,10.150386,2.262871e-01


In [12]:
def plot_histograms_long(df: pd.DataFrame) -> None:
	"""Plot histograms for each feature with different models superimposed.

	This function assumes the DataFrame is in long format with columns:
	"model", "feat_name", "feat_val", and optionally "prompt", "layer", "head".

	# Parameters:
	 - `df : pd.DataFrame`
		 DataFrame containing the data.

	# Returns:
	 - `None`
		 Displays the histograms.
	"""
	features: list[str] = df["feat_name"].unique().tolist()
	models: list[str] = df["model"].unique().tolist()

	for feature in features:
		plt.figure()
		subset_feature: pd.DataFrame = df[df["feat_name"] == feature]
		for model in models:
			subset_model: pd.DataFrame = subset_feature[
				subset_feature["model"] == model
			]
			plt.hist(
				subset_model["feat_val"], bins=50, alpha=0.5, label=model, density=True
			)
		plt.xlabel(feature)
		plt.ylabel("Frequency")
		plt.title(f"Histogram of {feature} for different models")
		plt.legend()
		plt.show()


plot_histograms_long(df)

KeyError: 'feat_name'